# Both Pick: data and exploratory analysis

How much rating history is available for two-person recommendations? This notebook checks the prepared MovieLens subset, then looks at ratings, coverage, genres and one pair of users. It uses recorded ratings, not model predictions.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import altair as alt
from src import data as d

pd.set_option("display.max_columns", 30, "display.width", 140)
# Okabe-Ito colour-blind-safe colours
BLUE, ORANGE = "#0072B2", "#E69F00"

ratings = d.load_ratings()
movies = d.load_movies()
genres = d.load_movie_genres()
config = d.load_json("sampling_config.json")
full_audit = d.load_json("audit_full_source.json")
sub_audit = d.load_json("audit_subset.json")
print(f"ratings {ratings.shape}, movies {movies.shape}, movie_genres {genres.shape}")

OUT = Path.cwd().parent / "outputs"  # regenerable exports (git-ignored)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(exist_ok=True)


def save_chart(chart, name):
    chart.save(str(OUT / "figures" / f"{name}.html"))
    return chart


def save_table(df, name):
    df.to_csv(OUT / "tables" / f"{name}.csv")


ratings (778056, 4), movies (23366, 6), movie_genres (47526, 2)


## 1. Data and preparation

MovieLens 32M is the source. The subset samples 5,000 users with seed 830 and keeps all their ratings. It represents MovieLens users, not all movie viewers.

One row in `ratings` is a recorded user–movie rating; `movies` has one row per movie; `movie_genres` has one row per movie–genre membership. Missing ratings are unobserved and are not filled with zeros. Release years are parsed from title suffixes and flagged when unavailable; missing genre labels are retained. Full-source and subset audits are reported separately.


In [2]:
coverage = pd.DataFrame(
    {
        "full source": [
            full_audit["ratings"]["distinct_users"],
            full_audit["ratings"]["ratings_rows"],
            full_audit["movies"]["movies_rows"],
        ],
        "subset": [ratings.userId.nunique(), len(ratings), len(movies)],
    },
    index=["users", "ratings", "movie metadata rows"],
)
display(coverage)
print(f"Sampling seed: {config['seed']}")
print(f"Observed user–movie cells: {len(ratings) / (ratings.userId.nunique() * len(movies)):.4%}")


,full source,subset
users,200948,5000
ratings,32000204,778056
movie metadata rows,87585,23366


Sampling seed: 830
Observed user–movie cells: 0.6660%


In [3]:
quality = pd.DataFrame(
    {
        "count": {
            "missing values (any column)": sum(sub_audit["null_counts"].values()),
            "exact duplicate rating rows": sub_audit["exact_duplicate_rows"],
            "duplicate user-movie pairs": sub_audit["duplicate_user_movie_pairs"],
            "ratings outside 0.5-5.0": sub_audit["ratings_out_of_range"],
            "ratings not in half-star steps": sub_audit["ratings_not_half_star_increment"],
            "timestamps outside 1995..2023-10-14": sub_audit["timestamps_outside_expected_range"],
            "ratings without movie metadata": sub_audit["ratings_without_movie_metadata"],
            "subset movies: release year missing/unparseable": sub_audit["movies_rows"]
            - sub_audit["movies_year_status_counts"].get("ok", 0),
            "subset movies: '(no genres listed)'": sub_audit["movies_no_genre_listed"],
        }
    }
)
quality["scope"] = "subset"
display(quality)
print("Full-source counterparts: see docs/data_quality.md (full-source table).")


,count,scope
missing values (any column),0,subset
exact duplicate rating rows,0,subset
duplicate user-movie pairs,0,subset
ratings outside 0.5-5.0,0,subset
ratings not in half-star steps,0,subset
timestamps outside 1995..2023-10-14,0,subset
ratings without movie metadata,0,subset
subset movies: release year missing/unparseable,77,subset
subset movies: '(no genres listed)',491,subset


Full-source counterparts: see docs/data_quality.md (full-source table).


## 2. Rating distribution

How are recorded ratings spread across the 0.5–5 scale? These are ratings of movies users chose to watch and rate, so the distribution is self-selected.


In [4]:
dist = (
    ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="n_ratings")
)
dist["share"] = dist["n_ratings"] / dist["n_ratings"].sum()
display(dist.assign(share=dist["share"].round(4)))
print(
    f"{len(ratings):,} ratings; mean {ratings.rating.mean():.3f}, median {ratings.rating.median():.1f}"
)
save_table(dist, "b_rating_distribution")
save_chart(
    alt.Chart(dist)
    .mark_bar(color=BLUE)
    .encode(
        x=alt.X("rating:O", title="Rating (stars)"),
        y=alt.Y("n_ratings:Q", title="Number of ratings"),
        tooltip=["rating", "n_ratings", alt.Tooltip("share:Q", format=".1%")],
    )
    .properties(
        title=f"Observed rating distribution (subset, n = {len(ratings):,} ratings)",
        width=500,
        height=280,
    ),
    "b_rating_distribution",
)


,rating,n_ratings,share
0,0.5,12171,0.0156
1,1.0,22161,0.0285
2,1.5,11897,0.0153
3,2.0,48164,0.0619
4,2.5,39878,0.0513
5,3.0,145167,0.1866
6,3.5,102505,0.1317
7,4.0,205873,0.2646
8,4.5,75643,0.0972
9,5.0,114597,0.1473


778,056 ratings; mean 3.564, median 4.0


alt.Chart(...)

## 3. Rating coverage

How much history supports each user and movie? Log scales make the skewed counts readable. Movie counts here cover the sampled users only.


In [5]:
per_user = ratings.groupby("userId").size().rename("n_ratings")
per_movie = ratings.groupby("movieId").size().rename("n_ratings")


def log_hist(counts, bins=30):
    edges = np.unique(np.logspace(0, np.log10(counts.max()) + 0.001, bins).round().astype(int))
    cut = pd.cut(counts, bins=edges, right=False)
    h = cut.value_counts().sort_index().rename_axis("bin").reset_index(name="n_entities")
    h["lo"] = h["bin"].map(lambda i: i.left)
    h["hi"] = h["bin"].map(lambda i: i.right)
    return h[["lo", "hi", "n_entities"]]


def hist_chart(counts, what, color):
    return (
        alt.Chart(log_hist(counts))
        .mark_bar(color=color)
        .encode(
            x=alt.X(
                "lo:Q", scale=alt.Scale(type="log"), title=f"Ratings per {what} (log scale, binned)"
            ),
            x2="hi:Q",
            y=alt.Y("n_entities:Q", title=f"Number of {what}s"),
            tooltip=[
                alt.Tooltip("lo:Q", title="from"),
                alt.Tooltip("hi:Q", title="up to (excl.)"),
                "n_entities",
            ],
        )
        .properties(title=f"Ratings per {what} (subset)", width=330, height=260)
    )


display(
    pd.DataFrame(
        {
            "per user": per_user.describe(percentiles=[0.1, 0.5, 0.9, 0.99]),
            "per movie": per_movie.describe(percentiles=[0.1, 0.5, 0.9, 0.99]),
        }
    ).round(1)
)
print("Movies with exactly 1 rating:", int((per_movie == 1).sum()), "of", len(per_movie))
save_table(
    pd.DataFrame({"per_user": per_user.describe(), "per_movie": per_movie.describe()}),
    "c_activity_summary",
)
save_chart(
    hist_chart(per_user, "user", BLUE) | hist_chart(per_movie, "movie", ORANGE),
    "c_ratings_per_user_and_movie",
)


,per user,per movie
count,5000.0,23366.0
mean,155.6,33.3
std,253.4,118.7
min,20.0,1.0
10%,25.0,1.0
50%,73.0,3.0
90%,349.1,66.0
99%,1256.0,573.0
max,4485.0,2576.0


Movies with exactly 1 rating: 8406 of 23366


alt.HConcatChart(...)

## 4. Movie averages and support

How does a movie’s mean rating vary with its rating count? Each heatmap cell counts movies. Small counts give little support for trusting a movie average.


In [6]:
ms = ratings.groupby("movieId")["rating"].agg(n="size", mean="mean").reset_index()
edges = np.unique(np.logspace(0, np.log10(ms.n.max()) + 0.001, 25).round().astype(int))
ms["n_bin"] = pd.cut(ms["n"], edges, right=False)
ms["mean_bin"] = (ms["mean"] / 0.25).apply(np.floor) * 0.25
cell = ms.groupby(["n_bin", "mean_bin"], observed=True).size().reset_index(name="movies")
cell["n_lo"] = cell["n_bin"].map(lambda i: i.left)
cell["n_hi"] = cell["n_bin"].map(lambda i: i.right)
cell["mean_hi"] = cell["mean_bin"] + 0.25
cell = cell.drop(columns="n_bin")
display(
    ms.groupby(
        pd.cut(ms["n"], [0, 1, 4, 19, 99, 10**6], labels=["1", "2-4", "5-19", "20-99", "100+"]),
        observed=True,
    )["mean"]
    .agg(movies="size", mean_of_means="mean", sd_of_means="std")
    .round(3)
)
save_table(cell, "e_movie_mean_vs_count_cells")
save_chart(
    alt.Chart(cell)
    .mark_rect()
    .encode(
        x=alt.X(
            "n_lo:Q", scale=alt.Scale(type="log"), title="Ratings received (log scale, binned)"
        ),
        x2="n_hi:Q",
        y=alt.Y(
            "mean_bin:Q", scale=alt.Scale(domain=[0.5, 5]), title="Movie mean rating (0.25 bins)"
        ),
        y2="mean_hi:Q",
        color=alt.Color("movies:Q", scale=alt.Scale(type="log", scheme="viridis"), title="Movies"),
        tooltip=["n_lo", "n_hi", "mean_bin", "movies"],
    )
    .properties(
        title=f"Movie mean rating vs. rating count ({len(ms):,} movies in subset)",
        width=520,
        height=320,
    ),
    "e_movie_mean_vs_count",
)


,movies,mean_of_means,sd_of_means
n,,,
1,8406,3.181,1.074
2-4,5352,3.227,0.769
5-19,4658,3.292,0.605
20-99,3216,3.314,0.542
100+,1734,3.544,0.415


alt.Chart(...)

## 5. Genre coverage

Which genres are represented? Movies can have several genres, so genre totals overlap. Counts come from movie-level summaries rather than multiplying individual rating rows. Genre frequency does not establish a user’s complete taste.


In [7]:
mstats = ratings.groupby("movieId")["rating"].agg(n="size", total="sum").reset_index()
mstats["movie_mean"] = mstats["total"] / mstats["n"]
gm = genres.merge(mstats, on="movieId", how="left", validate="many_to_one")
assert gm["n"].notna().all()
g = (
    gm.groupby("genre")
    .agg(
        movies=("movieId", "nunique"),
        ratings=("n", "sum"),
        rating_sum=("total", "sum"),
        movie_weighted_mean=("movie_mean", "mean"),
    )
    .reset_index()
)
g["rating_weighted_mean"] = g["rating_sum"] / g["ratings"]
g = g.sort_values("ratings", ascending=False)
display(g[["genre", "movies", "ratings", "rating_weighted_mean"]].round(3).reset_index(drop=True))

assert movies.movieId.nunique() == mstats.shape[0] == genres.movieId.nunique() and mstats[
    "n"
].sum() == len(ratings)
print("Base-table counts unchanged by genre expansion; genre rows:", len(genres))

order = g["genre"].tolist()
counts = (
    alt.Chart(g)
    .mark_bar(color=BLUE)
    .encode(
        y=alt.Y("genre:N", sort=order, title=None),
        x=alt.X("movies:Q", title="Movies with this genre"),
        tooltip=["genre", "movies", "ratings"],
    )
    .properties(title="Genre coverage (movies; overlapping)", width=300, height=380)
)
save_table(g.drop(columns="rating_sum"), "f_genre_summary")
save_chart(counts, "f_genre_coverage")


,genre,movies,ratings,rating_weighted_mean
0,Drama,10452,340782,3.710
1,Comedy,7748,273935,3.447
2,Action,3382,235414,3.502
3,Thriller,3921,209887,3.562
4,Adventure,2187,185642,3.543
5,Sci-Fi,1811,137124,3.513
6,Romance,3203,135643,3.563
7,Crime,2348,130543,3.720
8,Fantasy,1471,89632,3.528
9,Children,1410,67360,3.442


Base-table counts unchanged by genre expansion; genre rows: 47526


alt.Chart(...)

## 6. Two users’ recorded ratings

Compare ratings on movies both users rated. The example is the first eligible pair by ascending ID among the 200 lowest-ID users with at least 100 ratings and at least 20 shared movies. The pair is selected by coverage, not agreement.

This is a descriptive example, not held-out evaluation or evidence of enjoyment together. Correlation measures a linear association; mean absolute difference measures how far apart the recorded ratings are.


In [8]:
search = d.find_first_eligible_pair(ratings, min_ratings=100, pool_size=200, min_shared=20)
assert search["pair"] is not None, "No pair meets the stated coverage requirements."
ua, ub, n_shared = search["pair"]
print(f"Profiles {ua} and {ub}: {n_shared} shared movies")


Profiles 59 and 83: 56 shared movies


In [9]:
a = ratings[ratings.userId == ua][["movieId", "rating"]].rename(columns={"rating": "rating_a"})
b = ratings[ratings.userId == ub][["movieId", "rating"]].rename(columns={"rating": "rating_b"})
shared = a.merge(b, on="movieId", validate="one_to_one").merge(
    movies[["movieId", "title"]], on="movieId", validate="many_to_one"
)
shared["diff_a_minus_b"] = shared["rating_a"] - shared["rating_b"]
assert len(shared) == n_shared
corr = (
    np.corrcoef(shared.rating_a, shared.rating_b)[0, 1]
    if shared.rating_a.nunique() > 1 and shared.rating_b.nunique() > 1
    else float("nan")
)
summary = pd.Series(
    {
        "co-rated movies": len(shared),
        f"ratings by user {ua} (all)": int((ratings.userId == ua).sum()),
        f"ratings by user {ub} (all)": int((ratings.userId == ub).sum()),
        f"mean rating, user {ua} (shared)": shared.rating_a.mean(),
        f"mean rating, user {ub} (shared)": shared.rating_b.mean(),
        "share exactly equal": (shared.diff_a_minus_b == 0).mean(),
        "share within 0.5 star": (shared.diff_a_minus_b.abs() <= 0.5).mean(),
        "mean absolute difference": shared.diff_a_minus_b.abs().mean(),
        "mean signed difference (A-B)": shared.diff_a_minus_b.mean(),
        "Pearson correlation": corr,
    },
    name="value",
).round(3)
display(summary.to_frame())


grid = shared.groupby(["rating_a", "rating_b"]).size().reset_index(name="movies")
diag = pd.DataFrame({"x": [0.5, 5], "y": [0.5, 5]})
cells = (
    alt.Chart(grid)
    .mark_circle(color=BLUE, opacity=0.8)
    .encode(
        x=alt.X("rating_a:Q", scale=alt.Scale(domain=[0, 5.5]), title=f"User {ua} rating"),
        y=alt.Y("rating_b:Q", scale=alt.Scale(domain=[0, 5.5]), title=f"User {ub} rating"),
        size=alt.Size("movies:Q", title="Movies", scale=alt.Scale(range=[60, 600])),
        tooltip=["rating_a", "rating_b", "movies"],
    )
)
line = alt.Chart(diag).mark_line(color="grey", strokeDash=[4, 4]).encode(x="x:Q", y="y:Q")
agree = (cells + line).properties(
    title="Ratings on shared movies (dashed = equal)", width=330, height=330
)
dh = (
    shared.diff_a_minus_b.value_counts().sort_index().rename_axis("diff").reset_index(name="movies")
)
diffs = (
    alt.Chart(dh)
    .mark_bar(color=ORANGE)
    .encode(
        x=alt.X("diff:O", title=f"Rating difference (user {ua} minus user {ub})"),
        y=alt.Y("movies:Q", title="Shared movies"),
        tooltip=["diff", "movies"],
    )
    .properties(title="Rating differences", width=330, height=330)
)
save_table(summary.to_frame(), "g_pair_summary")
save_table(shared, "g_pair_shared_movies")
save_chart(agree | diffs, "g_pair_agreement")


,value
co-rated movies,56.000
ratings by user 59 (all),417.000
ratings by user 83 (all),150.000
"mean rating, user 59 (shared)",3.375
"mean rating, user 83 (shared)",4.143
share exactly equal,0.107
share within 0.5 star,0.125
mean absolute difference,1.393
mean signed difference (A-B),-0.768
Pearson correlation,-0.060


alt.HConcatChart(...)

## Interpretation to finish

Add your interpretation of the main patterns and explain what they imply for recommending movies to two people. The model and strategy comparisons are in notebook 02.
